# 📗 NeMo Guardrails 기초: Colang 문법과 대화 흐름(Flow) 설계

**NeMo Guardrails의 설계 철학** ([공식 문서](https://docs.nvidia.com/nemo/guardrails/), [Rebedea et al., 2023](https://arxiv.org/abs/2310.10501))
- **모델 밖에서, 실행할 때 지킨다:** 앱 코드와 LLM 사이에 둔 레일이 메시지가 오갈 때마다 규칙을 적용합니다. 모델에 학습된 안전장치 위에 한 겹 더하는 것입니다.
- **규칙은 개발자가 정한다(programmable):** 무엇을 막고 어떻게 답할지를 설정 파일과 Colang 규칙으로 정합니다. 모델을 다시 학습하지 않고 규칙만 바꿉니다.
- **모델과 분리된 규칙:** 레일이 특정 LLM에 묶이지 않아 모델을 바꿔도 같은 규칙을 씁니다. 다만 이 교안의 prompts.yml처럼 모델에 맞게 프롬프트를 손보기도 합니다.
- **사람이 읽을 수 있는 규칙(interpretable):** 규칙이 Colang 파일로 남으므로, 봇이 왜 그렇게 답했는지 규칙과 로그로 따라갈 수 있습니다.
- **대화 관리 방식:** 사용자 문장을 의도로 바꾸고 정해 둔 대화 경로를 따르게 하는 기존 챗봇의 대화 관리 방식을 LLM에 접목했습니다. 그래서 본인 확인 같은 정해진 업무 절차를 지키게 할 수 있습니다.

<img src="images/16_nemo_capabilities.png" width="1050" alt="NeMo Guardrails로 할 수 있는 일. 고객의 질문이 LLM을 거쳐 답변으로 돌아오는 경로를 NeMo Guardrails가 감싸고, 그 위에 레일 다섯 개를 둔다. 입력 레일은 위험한 질문을 차단하고, 대화 레일은 정해진 의도를 정해진 흐름으로 답하게 하며(이번 교안), 검색 레일은 검색 문서를, 출력 레일은 답변을 검사하고(다음 단원), 실행 레일은 도구 호출을 검사한다(그다음 단원). 개발자는 설정 폴더(config.yml, Colang 규칙, actions.py)로 이 레일들을 선언한다. 설정 파일과 Colang 규칙으로 LLM 앞뒤에 검사와 대화 흐름을 붙인다.">

**NeMo Guardrails란**
- NVIDIA의 오픈소스 파이썬 라이브러리(`nemoguardrails`)로, LLM 앞뒤에 레일을 붙여 입력·대화 흐름·검색·도구 호출·답변을 통제합니다.
- 레일을 앱 코드에 흩어 쓰지 않고 설정 폴더(config.yml과 Colang 규칙 파일)에 모아 선언합니다.
- **어디에 쓰나:** 공식 문서는 네 가지 쓰임을 듭니다.
    - 문서 기반 질의응답(RAG): 답이 문서와 맞는지 확인하고 답변을 검사합니다.
    - 업무 전용 도우미(챗봇): 정해진 주제를 벗어나지 않고, 설계한 대화 흐름대로 대화하게 합니다.
    - LLM API 서비스: 직접 운영하는 LLM 앞뒤에 검사를 붙입니다.
    - LangChain 체인: 이미 만든 체인을 감싸 레일을 더합니다(2-4).
- **가드레일만이 아니라 챗봇을 만드는 도구이기도 합니다:** 대화 레일은 "이 의도에는 이 순서로 처리하고 이 문장으로 답한다"를 정하는 기능이라, 로그인 확인 -> 조회 -> 답변처럼 정해진 업무 흐름을 가진 챗봇을 만들 수 있습니다. 문장까지 정해 둔 질문은 그 문장으로 답하고, 정해 두지 않은 부분만 LLM이 만듭니다.
- 이번 교안은 그중 대화 레일을 만듭니다.

<img src="images/10_lesson02_goals.png" width="1050" alt="이번 교안에서 배울 것: NeMo Guardrails로 대화 레일 만들기(config.yml과 Colang 규칙으로 정해진 질문에 정해진 흐름으로 답하기). 1 설정 폴더로 봇 실행(config.yml, rails.co), 2 메시지 처리 세 단계, 3 Colang으로 대화 레일 작성(user · bot · flow), 4 흐름 설계와 테스트 문장 검증.">

**학습 목표** (그림의 번호와 같습니다)

1. **설정 폴더를 `RailsConfig`로 읽고 `LLMRails`로 봇을 실행합니다.**
2. **메시지 처리 세 단계를 설명하고, 로그로 LLM 호출 수와 사용자 의도를 확인합니다.**
3. **Colang 문법(user·bot·flow, 변수·if·execute, 하위 흐름·stop)으로 대화 레일을 작성합니다.**
4. **의도 목록 -> 흐름 -> 테스트 문장 순서로 대화 흐름을 설계하고 검증한 뒤, 대화 레일이 막지 못하는 공격을 확인합니다.**

## 1. 설정 폴더와 실습 준비

NeMo Guardrails에서는 설정 폴더에 둔 파일이 곧 봇의 규칙입니다.

### 1-1. 설정 폴더와 Colang 버전

첫 설정 폴더 `data/본실습/rails_first/`에는 파일이 세 개 있습니다.

```
rails_first/
  config.yml   모델, 봇 설명, 예시 대화
  rails.co     Colang 규칙: 인사, 투자 추천 거절
  prompts.yml  의도 만들기·봇 메시지 프롬프트(이 교안의 모델에 맞게 고친 것)
```

**NeMo Guardrails 설정 폴더의 구성 요소**

| 파일 | 무엇을 정하나 | `rails_first/`에 | 이 교안에서 쓰는 방법 |
|---|---|---|---|
| `config.yml` | 모델(주 모델·임베딩), 봇 설명(instructions), 예시 대화(sample_conversation), 켤 레일 | 있음 | 읽어 보기만 하고 본실습의 모든 봇에 그대로 씁니다(6절 도우미는 `rails_hr/`의 config.yml) |
| `*.co` (Colang 파일) | 사용자 의도, 봇 메시지, 대화 흐름 | 있음(`rails.co`: 인사, 투자 추천 거절) | 2절 첫 봇에 씁니다. 3절부터는 노트북에 쓴 Colang 문자열을 `rails.co` 내용 뒤에 이어 붙입니다 |
| `prompts.yml` | 의도 만들기, 봇 메시지, 입력·출력 검사처럼 LLM에 맡기는 작업의 프롬프트 | 있음 | gpt-4.1-mini에 맞게 고쳐 둔 파일을 그대로 씁니다(1-3) |
| `actions.py` | 흐름에서 호출하는 파이썬 함수(액션) | 없음 | 함수를 노트북에서 정의하고 `register_action`으로 등록합니다(3-3) |
| `kb/` | 검색 레일이 읽을 문서 | 없음 | 검색 레일과 함께 다음 단원에서 씁니다 |

- **폴더로 읽기:** `RailsConfig.from_path(폴더 경로)`가 폴더 안의 .yml 파일을 합치고 .co 파일을 모두 읽습니다.
- **문자열로 읽기:** `RailsConfig.from_content(colang_content=..., yaml_content=...)`는 파일 없이 문자열로 같은 설정을 만듭니다. 문법 실습은 이 방법으로 짧은 Colang 코드를 바로 실행합니다.

**Colang 버전: 이 교안은 Colang 1.0을 씁니다**

Colang은 NeMo Guardrails의 대화 규칙 언어입니다. nemoguardrails 0.23.0의 기본값이 1.0이라 config.yml에 `colang_version`을 따로 적지 않습니다(2-1에서 출력해 확인합니다).

| 항목 | Colang 1.0 (이 교안) | Colang 2.0 (사용하지 않음) |
|---|---|---|
| 정의 | `define user`, `define bot`, `define flow` | `flow` 하나로 통일하고 `define`이 없음 |
| 분기 | `if` / `else if` / `else` | `if` / `elif`(또는 `else if`) / `else` |

- 공식 문서의 Colang 입문 예제는 2.0으로 시작합니다. 예제에 `define` 없는 `flow`, `elif`, `{$name}`이 보이면 2.0 문법이라 이 교안 코드에 넣지 않습니다.

### 1-2. 경로, API 키, 사용 통계 끄기

NeMo Guardrails는 config.yml의 모델을 직접 호출하므로 `ChatOpenAI` 객체 없이 경로와 API 키만 준비합니다. 익명 사용 통계를 보내지 않도록 라이브러리를 불러오기 전에 환경 변수를 설정합니다.

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# API 키 읽기: 단원 폴더 .env를 먼저 읽고, 없는 키만 저장소 최상위 .env에서 채웁니다
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

# NeMo Guardrails 가 익명 사용 통계를 NVIDIA 서버로 보내지 않게 합니다(라이브러리를 불러오기 전에 설정)
os.environ["NEMO_GUARDRAILS_NO_USAGE_STATS"] = "1"

rails_dir = data_dir / "본실습" / "rails_first"  # 첫 대화 레일의 설정 폴더
print("설정 폴더:", rails_dir)

### 1-3. 설정 파일 세 개 읽기

봇의 동작은 설정 파일 세 개가 모두 정하므로 실행 전에 먼저 읽습니다. Colang 문법 실습에서도 이 설정을 그대로 씁니다.

In [ ]:
# 첫 대화 레일의 설정 파일 세 개를 문자열로 읽습니다
config_yaml = (rails_dir / "config.yml").read_text(encoding="utf-8")
prompts_yaml = (rails_dir / "prompts.yml").read_text(encoding="utf-8")
first_colang = (rails_dir / "rails.co").read_text(encoding="utf-8")

# 문자열로 봇을 만들 때 넘길 설정: 폴더의 yml 두 개를 이어 붙이면 from_path 가 폴더를 읽은 것과 같습니다
rails_yaml = config_yaml + "\n" + prompts_yaml

print(config_yaml)

- **models:** `type: main`은 의도 만들기·다음 단계 고르기·봇 메시지 만들기에 쓰는 주 모델, `type: embeddings`는 예시 문장 검색에 쓰는 임베딩 모델입니다.
- **gpt-4.1-mini를 고른 이유:** NeMo는 의도 이름을 고를 때 temperature 0.001을 보내 응답이 덜 흔들리게 합니다. gpt-5.x 같은 추론 모델은 temperature를 받지 않아 이 효과가 없고, 호출마다 추론 시간이 더해집니다.
- **임베딩 모델:** NeMo의 기본 임베딩 모델은 영어용이라, 한국어 예시 문장을 검색할 수 있도록 OpenAI `text-embedding-3-small`로 설정했습니다.
- **instructions:** 모든 LLM 프롬프트 맨 앞에 들어가는 봇 설명입니다.
- **sample_conversation:** LLM 프롬프트마다 들어가는 예시 대화입니다. 기본값이 영어 대화라 한국어로 바꿨습니다.

In [ ]:
# Colang 규칙 파일: 인사와 투자 추천 거절
print(first_colang)

- **세 블록:** `define user`는 의도 이름(`express greeting`)과 예시 문장, `define bot`은 봇 메시지 이름과 정해 둔 문장, `define flow`는 의도가 들어오면 답할 순서입니다.
- **들여쓰기:** 블록 안의 줄은 들여씁니다(이 교안은 두 칸).

**prompts.yml: NeMo가 LLM에 보내는 프롬프트를 바꾸는 파일**
- **역할:** prompts.yml에 적은 작업(의도 만들기, 봇 메시지 만들기)은 NeMo 기본 프롬프트 대신 이 파일의 프롬프트로 LLM을 부릅니다. 적지 않은 다음 단계 고르기는 기본 프롬프트를 씁니다.
- **고친 이유:** 기본 프롬프트는 gpt-3.5·gpt-4용이라, gpt-4.1-mini는 정의한 의도가 있는데도 비슷한 새 이름을 지어내거나 답변에 형식 줄을 섞습니다.
- **고친 내용:** 프롬프트 끝에 "후보 의도와 맞으면 그 이름을, 아니면 새 영어 이름을 한 줄로 쓰라", "한국어 한 줄로 답하라"는 지시를 더했습니다. 이 교안이 쓰지 않는 검색 문서 자리는 봇 메시지 프롬프트에서 뺐습니다.
- **주의:** 프롬프트마다 `models: openai/gpt-4.1-mini`가 적혀 있어 config.yml의 주 모델에 맞춰 골라집니다. config.yml에서 모델을 바꾸면 이 이름도 같이 바꿉니다. 그대로 두면 오류 없이 기본 프롬프트가 쓰일 수 있습니다. 입력·출력 검사 같은 다른 레일을 켜면 그 작업의 프롬프트도 이 파일에 더합니다(다음 단원).

## 2. 첫 대화 레일 실행과 처리 순서

### 2-1. 설정 폴더로 봇 만들고 메시지 보내기

- **`RailsConfig`:** config.yml과 Colang 파일의 내용을 담는 설정 객체입니다.
- **`LLMRails`:** 설정대로 메시지를 처리하는 봇 객체입니다. 만들 때 config.yml의 모델을 준비합니다.
- **`generate_async`:** 메시지 목록(OpenAI 대화 형식의 `role`·`content`)을 받아 답변을 돌려주는 비동기 메서드입니다. 주피터 셀에서는 `await`로 바로 호출합니다.
- **동기 `generate()`는 쓰지 않습니다:** 주피터처럼 이벤트 루프가 이미 실행 중인 환경에서 호출하면 RuntimeError가 납니다.

In [ ]:
from nemoguardrails import LLMRails, RailsConfig

# 설정 폴더(config.yml, rails.co)를 읽어 설정 객체를 만듭니다
first_config = RailsConfig.from_path(str(rails_dir))

# 설정대로 메시지를 처리하는 봇을 만듭니다 (모델은 config.yml이 정합니다)
first_rails = LLMRails(first_config)
print("Colang 버전:", first_config.colang_version)

In [ ]:
# 인사 메시지를 보내고 답변 문장만 출력합니다
first_response = await first_rails.generate_async(messages=[{"role": "user", "content": "안녕하세요"}])
print(first_response["content"])

옵션 없이 호출하면 `{"role": "assistant", "content": ...}` 모양의 딕셔너리가 반환되고, 답변 문장은 `content`에 있습니다. 의도가 `express greeting`으로 잡히면 `define bot express greeting`에 적은 문장이 그대로 답변이 됩니다.

### 2-2. 메시지 처리 세 단계

<img src="images/11_nemo_three_steps.png" width="1050" alt="NeMo Guardrails의 메시지 처리 세 단계. 1 사용자 의도 만들기(LLM 호출), 2 다음 단계 고르기(맞는 흐름이 있으면 그대로, 없으면 LLM이 고른다), 3 봇 메시지 만들기(define bot 문장이 있으면 그대로, 없으면 LLM이 만든다). 흐름과 문장이 정해진 질문은 LLM 1회, 정해 둔 흐름이 없는 질문은 LLM 3회.">

처리 단계를 알아야 어느 단계에서 LLM을 호출하고 무엇을 고쳐야 결과가 바뀌는지 판단할 수 있습니다.

- **사용자 의도(canonical form, 정규형):** 사용자 문장을 `express greeting`처럼 짧은 영어 이름으로 바꾼 것입니다. 흐름은 문장이 아니라 이 이름으로 시작합니다.

1. **사용자 의도 만들기(`generate_user_intent`):** 임베딩으로 `define user` 예시 문장 가운데 질문과 가까운 5개를 골라 프롬프트에 넣고, LLM이 의도 이름을 만듭니다.
2. **다음 단계 고르기:** 의도 다음에 봇이 무엇을 할지(어떤 봇 메시지를 보낼지) 정합니다. 이때 흐름이 있는지를 봅니다.
    - **흐름(flow):** `define flow`로 "이 의도가 오면 봇은 이렇게 한다"를 순서대로 적어 둔 규칙입니다. `rails.co`의 투자 추천 흐름은 다음과 같습니다.
      ```
      define flow investment
        user ask investment recommendation
        bot refuse investment advice
      ```
    - **흐름이 있으면:** "펀드 추천해 주세요"가 `ask investment recommendation`으로 잡히면, 흐름의 다음 줄인 `bot refuse investment advice`가 다음 단계입니다. LLM을 호출하지 않습니다.
    - **흐름이 없으면:** "오늘 서울 날씨 어때요?"(2-3에서 보내는 질문)처럼 그 의도(`ask weather`)로 시작하는 흐름이 없으면, LLM이 `bot ...` 형태의 다음 봇 행동 이름을 만들어 고릅니다(`generate_next_steps`).
3. **봇 메시지:** 고른 봇 메시지에 `define bot` 문장이 있으면 그대로 씁니다(LLM 호출 없음). 없으면 LLM이 문장을 만듭니다(`generate_bot_message`).

| 질문 | 1 사용자 의도 | 2 다음 단계 | 3 봇 메시지 | LLM 호출 |
|---|---|---|---|---|
| 흐름과 문장이 정해진 질문(인사) | LLM | 흐름 | `define bot` 문장 | 1회 |
| 정해 둔 흐름이 없는 질문(날씨) | LLM | LLM | LLM | 3회 |

- **의도는 LLM이 정합니다:** 같은 문장도 실행마다 다른 의도로 분류될 수 있습니다. 이 교안의 테스트 결과가 실행마다 달라질 수 있는 이유입니다.
- **의도 이름이 같아야 흐름이 시작됩니다:** LLM이 만든 이름이 정의한 의도와 한 글자라도 다르면 정의하지 않은 질문처럼 처리됩니다. NeMo는 받은 이름을 정의한 의도 목록과 대조하거나 고쳐 주지 않습니다.

### 2-3. 로그로 LLM 호출 보기

답변 문자열만으로는 어느 단계를 거쳤는지 알 수 없으므로 생성 로그를 봅니다.

- **로그 옵션:** `options={"log": {"llm_calls": True, "internal_events": True}}`를 주면 반환값이 nemoguardrails의 `GenerationResponse` 객체로 바뀝니다.
- **`.response[0]["content"]`:** 답변 문장입니다.
- **`.log.llm_calls`:** LLM 호출 목록입니다. 항목마다 `.task`(작업 이름), `.prompt`, `.completion`(LLM 응답), `.total_tokens`가 있습니다.
- **`.log.internal_events`:** 처리 중에 생긴 이벤트(딕셔너리) 목록입니다. `type`이 `UserIntent`인 이벤트의 `intent`가 사용자 의도입니다.

문장을 보내고 로그를 읽는 코드는 교안 끝까지 반복해서 쓰므로 함수로 만듭니다. 앱이 넘기는 값(`context`)은 3-3에서 씁니다.

In [ ]:
async def ask(rails, text, context=None):
    """사용자 문장 하나를 보내고 로그(LLM 호출, 내부 이벤트)가 담긴 결과를 돌려줍니다."""
    messages = []

    # 앱이 넘기는 값(로그인한 고객 번호 등)이 있으면 context 메시지로 먼저 넣습니다
    if context is not None:
        messages.append({"role": "context", "content": context})
    messages.append({"role": "user", "content": text})

    # 로그 옵션을 주면 답변과 함께 LLM 호출 목록과 내부 이벤트를 돌려받습니다
    return await rails.generate_async(messages=messages, options={"log": {"llm_calls": True, "internal_events": True}})


def get_user_intent(result):
    """로그의 내부 이벤트에서 사용자 의도(UserIntent)를 꺼냅니다."""
    intent = None
    for event in result.log.internal_events:
        if event["type"] == "UserIntent":
            intent = event["intent"]
    return intent


def show_result(result):
    """사용자 의도, LLM 호출 작업 이름, 답변 문장을 출력합니다."""
    tasks = [call.task for call in result.log.llm_calls]
    print("사용자 의도:", get_user_intent(result))
    print(f"LLM 호출 {len(tasks)}회:", tasks)
    print("답변:", result.response[0]["content"])

In [ ]:
# 흐름과 정해 둔 문장이 모두 있는 인사를 보냅니다
greeting_result = await ask(first_rails, "안녕하세요")
show_result(greeting_result)

In [ ]:
# 어떤 흐름에도 정의되지 않은 질문을 보냅니다
weather_result = await ask(first_rails, "오늘 서울 날씨 어때요?")
show_result(weather_result)

두 결과의 LLM 호출 작업 이름을 2-2의 표와 비교합니다.

- **인사:** `generate_user_intent` 하나만 기록되었다면 흐름과 `define bot` 문장으로 끝난 것입니다.
- **날씨:** 의도가 정의한 두 의도(`express greeting`, `ask investment recommendation`)가 아니면 `generate_next_steps`가 더해지고, 고른 봇 메시지에 정해 둔 문장이 없으면 `generate_bot_message`까지 기록됩니다. 답변은 LLM이 instructions를 보고 만든 문장입니다.

날씨 질문은 LLM을 세 번 부릅니다(의도 이름 정하기 -> 다음 단계 고르기 -> 봇 답변 만들기). 호출마다 따로 프롬프트를 만들어 보내고, 앞 호출의 결과가 뒤 호출의 프롬프트에 들어갑니다. 세 호출의 프롬프트와 응답을 같은 방법으로 보도록 출력 함수를 하나 만듭니다.

In [ ]:
def print_llm_call(call):
    """LLM 호출 하나의 작업 이름, 프롬프트 메시지 전체, LLM 응답을 구분해 출력합니다."""
    print("작업:", call.task)

    # call.prompt 는 LLM 에 보낸 프롬프트 전체입니다. 메시지마다 "[cyan]역할[/]" 표시 뒤에 내용이 이어진 문자열이라,
    # 표시마다 잘라 (역할, 내용) 목록으로 바꿉니다 (맨 앞 조각은 표시 앞의 빈 줄이라 건너뜁니다)
    prompt_messages = []
    for part in call.prompt.split("[cyan]")[1:]:
        role, content = part.split("[/]", 1)
        prompt_messages.append((role, content.strip()))
    print("프롬프트 메시지 수:", len(prompt_messages))

    # 메시지마다 번호와 역할을 붙여 처음부터 끝까지 구분해 출력합니다
    for number, (role, content) in enumerate(prompt_messages, start=1):
        print(f"\n----- [{number}] {role} -----")
        print(content)

    # LLM 이 돌려준 답
    print("\n===== LLM 응답 =====")
    print(call.completion)

**첫 번째 호출: 의도 이름 정하기(`generate_user_intent`).** 이 호출의 출력은 답변이 아니라 의도 이름입니다.

In [ ]:
# 날씨 질문의 첫 번째 LLM 호출(의도 이름 정하기)
print_llm_call(weather_result.log.llm_calls[0])

- **[1]~[6] 봇 설명과 대화 예시:** [1]은 config.yml의 instructions, [2]~[6]은 sample_conversation 전체입니다.
- **[7]~[17] 의도 예시:** `rails.co`의 예시 문장 6개 가운데 질문과 가까운 5개가 User 메시지로, 그 의도 이름이 바로 다음 Bot 메시지(`User intent: 이름`)로 들어갑니다. User·Bot 5쌍입니다.
- **[18]~[22]:** 실제 대화가 아니라 sample_conversation의 앞 두 턴으로, 형식을 보여 주는 예시입니다.
- **[23] 실제 대화:** `ask`는 질문마다 새 대화로 보내므로 이번 질문 하나뿐입니다.
- **[24] 마지막 지시:** prompts.yml이 더한 지시입니다. 후보 의도와 맞으면 그 이름을, 아니면 새 영어 이름을 쓰게 합니다. NeMo는 LLM 응답 `User intent: ...`에서 의도 이름을 꺼냅니다.

**두 번째 호출: 다음 단계 고르기(`generate_next_steps`).** 첫 번째 호출이 정한 의도로 시작하는 흐름이 없어서, 봇이 다음에 할 행동을 LLM이 고릅니다.

In [ ]:
# 날씨 질문의 두 번째 LLM 호출(다음 단계 고르기)
print_llm_call(weather_result.log.llm_calls[1])

- **[1]~[6]:** 첫 번째 호출과 같은 봇 설명과 대화 예시입니다.
- **[7]~[9] 흐름 예시(`This is how the bot thinks:`):** `rails.co`의 흐름이 "의도 -> 봇 행동" 쌍으로 들어갑니다. 흐름이 둘(인사, 투자 추천)이라 두 쌍입니다.
- **[10]~[16] 지금 대화:** 형식 예시 두 턴 뒤에 이번 질문과, 첫 번째 호출이 정한 의도(`User intent: ...`)가 이어집니다. 앞 호출의 결과가 이렇게 다음 호출의 프롬프트에 들어갑니다.
- **LLM 응답:** NeMo는 응답 첫 줄이 [8]·[9]처럼 `Bot intent: 이름` 형식이면 그 이름을 다음 봇 행동으로 쓰고, 그렇지 않으면 `general response`(일반 응답)로 처리합니다. 이 호출은 prompts.yml로 고치지 않은 NeMo 기본 프롬프트를 써서, gpt-4.1-mini가 봇 행동 이름 대신 답변 문장을 바로 쓰기도 합니다. 이런 응답 문장은 NeMo가 버리므로 세 번째 호출은 이 문장을 보지 않습니다.

**세 번째 호출: 봇 답변 만들기(`generate_bot_message`).** 두 번째 호출이 고른 봇 메시지에 `define bot` 문장이 없어서, 사용자에게 보낼 문장을 LLM이 만듭니다.

In [ ]:
# 날씨 질문의 세 번째 LLM 호출(봇 답변 만들기)
print_llm_call(weather_result.log.llm_calls[2])

# 이 호출의 응답이 사용자에게 간 최종 답변입니다
print("\n===== 사용자에게 간 답변 =====")
print(weather_result.response[0]["content"])

- **[8]~[12] 봇 말투 예시(`This is how the bot talks:`):** 다음 봇 행동 이름과 이름이 가까운 `define bot` 5개가 문장과 함께 들어갑니다. `rails.co`의 문장뿐 아니라 NeMo가 기본으로 불러오는 내장 규칙의 영어 문장(예: `refuse to respond`)도 후보라서 함께 들어올 수 있습니다.
- **[13]~[19] 지금 대화:** 이번 질문, 첫 번째 호출이 정한 의도, 두 번째 호출이 정한 봇 행동(`Bot intent: ...`)까지 이어집니다.
- **[20] 마지막 지시:** prompts.yml이 더한 지시(한국어 한 줄로 답하라)입니다.
- **LLM 응답:** NeMo가 `Bot message: "..."`에서 문장만 꺼내 사용자에게 보냅니다. 맨 아래 "사용자에게 간 답변"과 같은 문장입니다.

### 2-4. 기존 LangChain 체인에 붙이기

이미 LangChain으로 만든 앱이 있으면 봇을 새로 만들지 않고, 같은 설정 폴더를 `RunnableRails`로 감싸 체인 앞에 붙일 수 있습니다.

- **`RunnableRails(설정) | 체인`:** 보통의 `|`처럼 앞뒤를 차례로 실행하지 않습니다. 정해 둔 흐름이 있는 질문(인사, 투자 추천)은 레일이 정해 둔 문장으로 답하고, 레일이 답변 문장을 만들어야 할 때(여기서는 흐름이 없는 질문)만 뒤의 체인(`support_chain`)이 답합니다.
- **반환값은 두 가지입니다:** 레일이 답하면 `{"output": 답변}` 딕셔너리, 체인이 답하면 체인의 출력(여기서는 문자열)이 그대로 나옵니다.
- **호출:** 2-1과 같은 이유로 주피터에서는 `await ... .ainvoke({"input": 질문})`으로 호출합니다.
- 이 교안의 나머지 실습은 단계별 로그를 보려고 `LLMRails`를 직접 씁니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from nemoguardrails.integrations.langchain.runnable_rails import RunnableRails

# 기존 LangChain 체인: 프롬프트 -> 모델 -> 문자열 (레일 없음)
support_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 한빛카드 고객센터 상담 챗봇이다. 한국어로 짧게 답한다."),
    ("user", "{input}"),
])
support_chain = support_prompt | ChatOpenAI(model="gpt-4.1-mini") | StrOutputParser()

# 첫 대화 레일의 설정 폴더를 RunnableRails 로 감싸 체인 앞에 붙입니다
guarded_chain = RunnableRails(RailsConfig.from_path(str(rails_dir))) | support_chain

In [ ]:
# 흐름이 있는 질문(투자 추천)과 흐름이 없는 질문(연회비)을 보내 누가 답하는지 확인합니다
for question in ["펀드 추천해 주세요", "연회비는 언제 청구되나요?"]:
    guarded_answer = await guarded_chain.ainvoke({"input": question})
    print(question, "->", guarded_answer)

**결과 읽기**
- **투자 추천:** 레일의 정해 둔 거절 문장이 `{"output": ...}`으로 나옵니다. LLM 호출은 의도 만들기 1회뿐이고, `support_chain`은 실행되지 않습니다.
- **연회비:** 흐름이 없어 `support_chain`이 만든 문자열이 나옵니다. LLM 호출은 3회입니다(NeMo의 의도 만들기·다음 단계 고르기 2회 + `support_chain` 1회). 레일을 붙이면 체인만 쓸 때(1회)보다 호출이 늘어납니다.
- **대화 레일은 흐름을 정해 둔 질문만 통제합니다:** 연회비 답은 검색 문서 없이 모델이 만든 문장이라 사실과 다를 수 있습니다.

## 3. Colang 문법: 흐름을 하나씩 추가하기

첫 봇은 인사와 투자 추천 거절뿐이었습니다. 소절마다 첫 봇의 Colang(`first_colang`)에 문법 요소 하나를 추가해 실행하고, 3-4는 3-3의 배송 조회 흐름에 로그인 확인을 더합니다.

**Colang 블록 네 가지**

| 블록 | 뜻 | 예 |
|---|---|---|
| `define user` | 사용자 의도와 그 예시 문장 | `define user express greeting` |
| `define bot` | 봇 메시지와 정해 둔 문장 | `define bot express greeting` |
| `define flow` | 의도가 들어오면 봇이 할 일의 순서 | `define flow greeting` |
| `define subflow` | 다른 흐름이 `do`로 호출하는 하위 흐름 | `define subflow check login` |

### 3-1. define user: 의도와 예시 문장

예시 문장을 어떻게 쓰느냐가 의도 만들기의 정확도를 정합니다.

- **예시 문장은 정답 목록이 아닙니다:** 의도 만들기 프롬프트에 들어가는 견본입니다. 질문과 글자가 같지 않아도 LLM이 뜻으로 의도를 고릅니다.
- **가까운 5개만 들어갑니다:** 의도마다 표현이 다른 문장을 여러 개 두어야 바꿔 말한 질문에도 그 의도의 예시가 프롬프트에 들어갑니다.
- **확인은 예시에 없는 문장으로 합니다:** 예시 문장을 그대로 넣으면 일치하는 것이 당연해서 검증이 되지 않습니다.

`rails.co`에 분실 신고 의도와 흐름을 더합니다. Colang 문자열을 이어 붙이면 파일 여러 개를 함께 읽은 것과 같습니다.

<img src="images/13_colang_blocks.png" width="1050" alt="Colang 세 블록: 의도 -> 흐름 -> 봇 메시지. 고객의 '어제 택시에서 카드를 흘린 것 같아요'를 define user의 예시 문장(카드를 잃어버렸어요, 카드 분실 신고하고 싶어요, 카드가 없어졌어요)과 비교해 LLM이 가까운 예시를 보고 의도 이름 report lost card를 고른다. define flow가 user report lost card 다음에 bot inform lost card를 정하고, define bot의 정해 둔 문장 '앱 [카드 관리 > 분실 신고]에서 바로 정지할 수 있습니다'가 LLM을 호출하지 않고 답변이 된다.">

In [ ]:
# 분실 신고 의도와 흐름
LOST_COLANG = """
# 분실 신고: 고객이 카드를 잃어버렸다고 하면 정해 둔 안내 문장으로 답한다
define user report lost card
  "카드를 잃어버렸어요"
  "카드 분실 신고하고 싶어요"
  "카드가 없어졌어요"

define bot inform lost card
  "앱 [카드 관리 > 분실 신고]에서 바로 사용을 정지할 수 있습니다. 전화 신고는 1599-0000(24시간)으로 해 주세요."

define flow lost card
  user report lost card
  bot inform lost card
"""

# 인사, 투자 추천 거절(rails.co)에 분실 신고를 더해 봇을 만듭니다 (설정은 rails_yaml 그대로)
lost_rails = LLMRails(RailsConfig.from_content(colang_content=first_colang + LOST_COLANG, yaml_content=rails_yaml))

# 예시 문장에 없는 바꿔 말한 문장을 보냅니다
lost_result = await ask(lost_rails, "어제 택시에서 카드를 흘린 것 같아요")
show_result(lost_result)

"흘린 것 같아요"는 예시 문장에 없는 표현입니다. 의도가 `report lost card`로 나오면 LLM이 뜻으로 분류한 것이고, 다른 의도가 나오면 이런 표현의 예시 문장을 더합니다.

**흐름이 여러 개면 의도가 곧 분기입니다**
- NeMo는 사용자 의도로 시작하는 흐름을 찾아 실행하므로 if 문으로 흐름을 고를 필요가 없습니다. `lost_rails`에서 의도가 `report lost card`로 잡히면 인사·투자 추천 거절 흐름과 상관없이 분실 흐름만 실행됩니다.
- 새 질문을 안내하거나(분실 신고처럼 처리할 질문) 정해 둔 문장으로 거절하려면(투자 추천처럼 답하면 안 되는 요청) `define user`, `define bot`, `define flow` 세 블록을 함께 더합니다. 파이썬 코드는 바꾸지 않습니다.

### 3-2. define bot: 정해 둔 문장과 LLM이 만드는 문장

- **문장이 여러 개면:** 그중 하나를 무작위로 고릅니다.
- **문장을 정의하지 않으면:** 흐름에 적은 봇 메시지 이름(`bot explain card benefits`)만 보고 LLM이 문장을 만듭니다. 흐름이 있으므로 다음 단계 고르기는 LLM을 호출하지 않고, 봇 메시지에서 호출이 1회 더해집니다.

In [ ]:
# 문장이 두 개인 봇 메시지와, 문장을 정의하지 않은 봇 메시지
BOT_MESSAGE_COLANG = """
define user express thanks
  "고마워요"
  "감사합니다"

# 정해 둔 문장이 여러 개면 그중 하나를 무작위로 고른다
define bot express thanks
  "도움이 되어 기쁩니다."
  "언제든 다시 문의해 주세요."

define flow thanks
  user express thanks
  bot express thanks

define user ask card benefits
  "이 카드 혜택이 뭐예요?"
  "카드 쓰면 뭐가 좋아요?"

# bot explain card benefits 는 define bot 이 없으므로 LLM이 문장을 만든다
define flow card benefits
  user ask card benefits
  bot explain card benefits
"""

# 인사, 투자 추천 거절(rails.co)에 감사, 카드 혜택 흐름을 더해 봇을 만듭니다
bot_message_rails = LLMRails(RailsConfig.from_content(colang_content=first_colang + BOT_MESSAGE_COLANG, yaml_content=rails_yaml))

# 같은 감사 인사를 세 번 보내 어떤 문장이 나오는지 봅니다
thanks_results = []
for count in range(3):
    thanks_result = await ask(bot_message_rails, "정말 고맙습니다")
    thanks_results.append({
        "회차": count + 1,
        "사용자 의도": get_user_intent(thanks_result),
        "답변": thanks_result.response[0]["content"],
    })
display(pd.DataFrame(thanks_results))

In [ ]:
# define bot 문장이 없는 봇 메시지로 가는 질문을 보냅니다
benefits_result = await ask(bot_message_rails, "이 카드는 어떤 점이 좋나요?")
show_result(benefits_result)

- **감사 인사:** 의도가 `express thanks`이면 답변 열에 정해 둔 두 문장 가운데 하나가 나옵니다. 세 번 모두 같은 문장이 나올 수도 있습니다(무작위 선택).
- **카드 혜택:** 의도가 `ask card benefits`이면 `generate_bot_message`가 더해지고 답변은 LLM이 만든 문장입니다. 봇 설명을 따라 고객센터로 안내하기도 하지만, 카드 혜택 자료가 없으므로 실행마다 문장이 달라지고 혜택을 지어낼 수도 있습니다.
- **그래서 거절·안내 문구는 `define bot`으로 고정합니다:** 문구가 정확해야 하는 메시지를 LLM이 생성하게 하면 공격 문장이나 모델의 판단에 따라 내용이 달라집니다.

### 3-3. 변수, execute, if: 조회 결과로 답 고르기

배송 조회처럼 고객마다 답이 다른 질문은 흐름 안에서 조회 결과를 변수에 담아 답을 고릅니다.

- **변수:** `$`로 시작하는 이름입니다(`$status`). 모든 흐름이 공유합니다.
- **context 메시지:** 앱이 `{"role": "context", "content": {"customer_id": "C001"}}`을 사용자 메시지 앞에 넣으면 흐름에서 `$customer_id`로 읽습니다. NeMo는 `$` 뒤의 이름과 같은 키를 context에서 찾으므로 두 이름이 글자까지 같아야 하고, 키가 없으면 `if` 문에서 None으로 읽힙니다(3-4의 로그인 확인이 이를 이용합니다). context 메시지를 여러 번 넣으면 값이 합쳐지고, 같은 키는 나중에 넣은 값으로 바뀝니다.
- **고객 번호는 앱이 넘깁니다:** 사용자가 말한 이름이나 번호로 조회하면 본인 사칭(남이 본인이라고 말하기)을 막을 방법이 없습니다.
- **액션:** 흐름에서 호출하는 파이썬 함수입니다. `rails.register_action(함수, "이름")`으로 등록하고, `$status = execute get_delivery_status(customer_id=$customer_id)`처럼 인자를 키워드로 넘겨 결과를 변수에 담습니다. LLM이 부를지 정하는 도구(tool)와 달리, 흐름이 정해진 자리에서 부릅니다.
- **`if` / `else if` / `else`:** 변수 값으로 봇 메시지를 고릅니다. `elif`는 Colang 1.0에서 파싱 오류입니다.

실제 서비스라면 액션이 카드사 시스템을 조회합니다. 이 교안은 미리 만들어 둔 `customers.json`을 읽어 그 역할을 대신합니다.

In [ ]:
# 카드사 시스템 대신 쓰는 고객 데이터 (고객 번호 -> 카드 배송 상태, 사용 가능 포인트)
customers = json.loads((data_dir / "본실습" / "customers.json").read_text(encoding="utf-8"))
display(customers)

In [ ]:
def get_delivery_status(customer_id):
    """고객 번호로 카드 배송 상태를 조회합니다."""
    return customers[customer_id]["delivery"]


# 배송 조회 의도와 안내 문장 세 개 (흐름은 아래에 따로 둡니다. 다음 소절에서 흐름만 바꿔 씁니다)
DELIVERY_MESSAGES = """
define user ask delivery status
  "카드 언제 와요?"
  "배송 조회해 주세요"
  "발급 신청한 카드가 어디쯤 있나요?"

define bot inform delivery shipping
  "카드가 배송 중입니다. 1~2영업일 안에 도착합니다."

define bot inform delivery review
  "카드 발급 심사 중입니다. 심사가 끝나면 문자로 알려 드립니다."

define bot inform delivery other
  "카드 상태는 상담원 확인이 필요합니다. 고객센터 1599-0000으로 문의해 주세요."
"""

# 배송 상태를 조회해 $status 에 담고, 값에 따라 안내 문장을 고르는 흐름
DELIVERY_FLOW = """
define flow delivery status
  user ask delivery status
  $status = execute get_delivery_status(customer_id=$customer_id)
  if $status == "배송 중"
    bot inform delivery shipping
  else if $status == "발급 심사 중"
    bot inform delivery review
  else
    bot inform delivery other
"""
delivery_rails = LLMRails(RailsConfig.from_content(colang_content=first_colang + DELIVERY_MESSAGES + DELIVERY_FLOW, yaml_content=rails_yaml))

# Colang의 execute 이름과 같은 이름으로 액션을 등록합니다
delivery_rails.register_action(get_delivery_status, "get_delivery_status")

# 등록된 액션 목록에 이름이 들어갔는지 확인합니다
print("get_delivery_status 등록:", "get_delivery_status" in delivery_rails.runtime.action_dispatcher.registered_actions)

In [ ]:
# 같은 배송 문의를 고객 세 명의 번호로 보내 if 문의 세 분기를 모두 확인합니다
delivery_results = []
for customer_id in ["C001", "C002", "C003"]:
    delivery_result = await ask(delivery_rails, "제 카드 어디쯤 왔어요?", {"customer_id": customer_id})
    delivery_results.append({
        "고객 번호": customer_id,
        "사용자 의도": get_user_intent(delivery_result),
        "LLM 호출 수": len(delivery_result.log.llm_calls),
        "답변": delivery_result.response[0]["content"],
    })
display(pd.DataFrame(delivery_results))

- **세 분기:** 의도가 `ask delivery status`로 잡히면 같은 질문이라도 `$status`에 따라 고객마다 다른 정해 둔 문장이 나옵니다.
- **LLM 호출:** 의도가 잡히면 조회와 분기는 파이썬 함수와 Colang이 처리하므로, LLM 호출은 의도 만들기 한 번입니다.
- **로그인하지 않았다면:** context 없이 보내도 조회부터 합니다. 조회 함수가 고객 번호 대신 `"$customer_id"`라는 글자를 받아 오류가 나고, NeMo는 영어 내부 오류 문장("I'm sorry, an internal error has occurred.")으로 답합니다.

### 3-4. 하위 흐름과 stop: 같은 흐름에 로그인 확인 더하기

<img src="images/14_flow_login_branch.png" width="1050" alt="변수 · execute · if와 하위 흐름: 배송 조회. 앱이 넘긴 context(customer_id = C001)와 고객 질문 '제 카드 어디쯤 왔어요?'가 flow delivery status로 들어간다. do check login에서 customer_id가 없으면 bot ask login 뒤 stop으로 멈추고, 있으면 $status = execute get_delivery_status로 조회한 뒤 if $status가 배송 중이면 bot inform delivery shipping, else if 발급 심사 중이면 bot inform delivery review, else이면 bot inform delivery other로 답한다. stop은 하위 흐름 안에서 써도 호출한 흐름까지 멈춘다.">

메시지 정의는 그대로 두고 배송 조회 흐름의 조회 앞에 로그인 확인을 더합니다.

**하위 흐름**(subflow)은 여러 흐름이 같이 쓰는 단계를 따로 떼어 이름을 붙인 흐름입니다. 사용자 의도로는 시작되지 않고, 다른 흐름이 `do 이름`으로 부를 때만 실행된 뒤 부른 흐름의 다음 줄로 돌아갑니다. 파이썬에서 반복되는 코드를 함수로 빼고 호출하는 것과 같습니다. 로그인 확인은 포인트 조회 같은 다른 조회 흐름에도 필요하므로 하위 흐름으로 만듭니다.

아래 코드에서는 다음과 같이 씁니다.

- **`define subflow check login`:** 로그인 확인 하위 흐름입니다. 배송 조회 흐름이 조회 바로 앞에서 `do check login`으로 부릅니다.
- **`if not $customer_id`:** 앱이 context로 고객 번호를 넘기지 않았으면 참이 되어 `bot ask login`으로 로그인을 안내합니다.
- **`stop`:** 안내한 자리에서 대화 처리를 멈춥니다. 하위 흐름 안에 있지만 부른 배송 조회 흐름까지 멈추므로 뒤의 `execute get_delivery_status`가 실행되지 않습니다. 고객 번호가 있으면 하위 흐름은 아무것도 하지 않고 끝나고, 배송 조회 흐름이 조회로 넘어갑니다.

In [ ]:
# 로그인 확인 하위 흐름: 고객 번호가 없으면 안내하고 흐름 전체를 멈춥니다
LOGIN_COLANG = """
define bot ask login
  "로그인한 고객만 조회할 수 있습니다. 앱에서 로그인한 뒤 다시 문의해 주세요."

define subflow check login
  if not $customer_id
    bot ask login
    stop
"""

# 배송 조회 흐름: 앞 소절의 흐름에서 조회 앞에 do check login 한 줄만 더했습니다
DELIVERY_LOGIN_FLOW = """
define flow delivery status
  user ask delivery status
  do check login
  $status = execute get_delivery_status(customer_id=$customer_id)
  if $status == "배송 중"
    bot inform delivery shipping
  else if $status == "발급 심사 중"
    bot inform delivery review
  else
    bot inform delivery other
"""

# 메시지 정의(DELIVERY_MESSAGES)는 그대로 두고 흐름만 바꿔 봇을 다시 만듭니다
login_rails = LLMRails(RailsConfig.from_content(colang_content=first_colang + DELIVERY_MESSAGES + LOGIN_COLANG + DELIVERY_LOGIN_FLOW, yaml_content=rails_yaml))
login_rails.register_action(get_delivery_status, "get_delivery_status")

# 등록된 액션 목록에 이름이 들어갔는지 확인합니다
print("get_delivery_status 등록:", "get_delivery_status" in login_rails.runtime.action_dispatcher.registered_actions)

In [ ]:
# 로그인한 고객 C001로 질문을 보냅니다
login_result = await ask(login_rails, "제 카드 어디쯤 왔어요?", {"customer_id": "C001"})
show_result(login_result)
print()

# 같은 질문을 context 없이(로그인하지 않은 상태로) 보냅니다
no_login_result = await ask(login_rails, "제 카드 어디쯤 왔어요?")
show_result(no_login_result)

In [ ]:
# 로그인하지 않은 실행의 내부 이벤트에서 실행된 액션 이름을 모아 조회 액션이 실행됐는지 봅니다
executed_actions = []
for event in no_login_result.log.internal_events:
    if event["type"] == "StartInternalSystemAction":
        executed_actions.append(event["action_name"])
display(executed_actions)
print("get_delivery_status 실행 여부:", "get_delivery_status" in executed_actions)

`generate_user_intent`처럼 NeMo가 내부적으로 실행하는 액션은 목록에 있지만, 의도가 `ask delivery status`로 잡혔다면 `stop`에서 멈췄으므로 `get_delivery_status`는 목록에 없습니다. `generate_bot_message` 액션은 정해 둔 문장을 꺼낼 때도 실행되므로, LLM을 불렀는지는 `llm_calls`로 봅니다.

## 4. 대화 흐름 설계

### 4-1. 설계 순서

의도를 빠뜨리거나 예시 문장이 부족하면 대화 레일이 설계와 다른 흐름을 실행하므로, 정해진 순서로 설계하고 테스트 문장으로 검증합니다.

<img src="images/12_flow_design_steps.png" width="1050" alt="대화 흐름 설계: 의도 목록에서 테스트까지. 1 의도 목록(처리 · 거절 · 전환), 2 흐름 정하기(로그인 확인 -> 조회), 3 Colang 작성(define user · flow), 4 테스트 문장(예시에 없는 문장), 5 결과 표(기대 의도와 실제 의도), 6 예시 문장 보강(비슷한 표현 추가). 결과가 어긋나면 4 테스트 문장으로 돌아가 다시 확인한다.">

1. **의도 목록:** 처리할 의도, 거절할 의도, 사람에게 넘길 의도(전환)로 나눕니다. 교안 01의 위협 분석 표에서 대화 레일로 막기로 한 것이 거절 의도가 됩니다.
2. **흐름 정하기:** 의도마다 봇이 할 일을 정합니다. 조회가 필요하면 로그인 확인 -> 조회 -> 상태별 안내처럼 단계를 적습니다.
3. **Colang 작성:** 의도마다 표현이 다른 예시 문장을 여러 개 두고(3-1), 거절·안내 문구는 `define bot`으로 고정합니다(3-2).
4. **테스트 문장:** 예시 문장과 겹치지 않게 바꿔 말한 문장과 범위 밖 문장을 준비합니다.
5. **결과 표:** 문장마다 기대 의도와 실제 의도를 나란히 놓습니다.
6. **예시 문장 보강:** 어긋난 문장의 기대 의도에 예시 문장을 더하고 다시 테스트합니다(방법은 4-3).

### 4-2. 한빛카드 의도 목록과 전체 Colang

| 종류 | 의도 | 흐름이 하는 일 |
|---|---|---|
| 처리 | `express greeting` | 인사말 |
| 처리 | `report lost card` | 분실 신고 방법 안내(정해 둔 문장) |
| 처리 | `ask delivery status` | 로그인 확인 -> 배송 상태 조회 -> 상태별 안내 |
| 처리 | `ask my points` | 로그인 확인 -> 포인트 조회 -> 포인트 안내 |
| 거절 | `ask investment recommendation` | 투자 상품 추천 거절 |
| 거절 | `ask other customer info` | 다른 고객 정보 조회 거절(교안 01의 남의 정보 조회) |
| 전환 | `request human agent` | 상담원 연결 안내 |

- **의도 이름:** 4-3의 테스트 문장 파일 `flow_tests.json`의 기대 의도와 글자까지 같게 씁니다.
- **새로 쓰는 것:** 2·3절의 흐름(3-2의 감사·카드 혜택은 문법 예시라 제외)은 그대로 이어 붙이고, 포인트 조회·다른 고객 정보 거절·상담원 연결만 `NEW_INTENTS_COLANG`으로 씁니다.

**포인트 조회: 조회 결과를 봇 메시지 안에 넣기**
- 배송 조회와 같은 순서(로그인 확인 -> 조회 -> 안내)로 짜고, 하위 흐름 `check login`을 다시 씁니다.
- 조회한 포인트는 `"사용 가능한 포인트는 $points 포인트입니다."`처럼 변수 뒤를 띄어 쓰거나 `{{ points }}`로 넣습니다.
- **`$변수` 뒤에 한글을 붙이면:** `$points포인트입니다.`는 띄어쓰기나 마침표 앞까지(`points포인트입니다`)를 변수 이름으로 읽어, 그 부분 전체가 빈 문자열이 됩니다.

In [ ]:
def get_my_points(customer_id):
    """고객 번호로 사용 가능한 포인트를 조회합니다."""
    return customers[customer_id]["points"]

In [ ]:
# 이 절에서 새로 쓰는 의도 셋: 포인트 조회, 다른 고객 정보 거절, 상담원 연결
NEW_INTENTS_COLANG = """
# 포인트 조회: 로그인 확인 -> 포인트 조회 -> 포인트 안내
define user ask my points
  "포인트 얼마나 있어요?"
  "내 포인트 조회해 주세요"
  "적립된 포인트 확인하고 싶어요"

define bot inform points
  "사용 가능한 포인트는 $points 포인트입니다."

define flow my points
  user ask my points
  do check login
  $points = execute get_my_points(customer_id=$customer_id)
  bot inform points

# 다른 고객 정보 조회 거절
define user ask other customer info
  "다른 사람 카드 정보 좀 알려 주세요"
  "남편 카드 사용 내역 조회해 주세요"
  "다른 고객 연락처 좀 알려 주세요"

define bot refuse other customer info
  "죄송하지만 다른 고객의 정보는 알려 드릴 수 없습니다. 카드 명의자 본인이 로그인해서 조회해 주세요."

define flow other customer info
  user ask other customer info
  bot refuse other customer info

# 상담원 연결
define user request human agent
  "상담원 연결해 주세요"
  "상담사 바꿔 주세요"
  "ARS 말고 상담원이요"

define bot transfer to agent
  "상담원에게 연결해 드리겠습니다. 평일 9시부터 18시까지 연결되며, 그 밖의 시간에는 1599-0000으로 문의해 주세요."

define flow human agent
  user request human agent
  bot transfer to agent
"""

# 2, 3절에서 만든 Colang(인사, 투자 추천 거절, 분실 신고, 배송 조회 메시지, 로그인 확인, 배송 조회 흐름)에 새 의도 셋을 이어 붙입니다
CARD_COLANG = first_colang + LOST_COLANG + DELIVERY_MESSAGES + LOGIN_COLANG + DELIVERY_LOGIN_FLOW + NEW_INTENTS_COLANG

In [ ]:
# 전체 Colang으로 봇을 만들고 조회 액션 두 개를 등록합니다
card_config = RailsConfig.from_content(colang_content=CARD_COLANG, yaml_content=rails_yaml)
card_rails = LLMRails(card_config)
card_rails.register_action(get_delivery_status, "get_delivery_status")
card_rails.register_action(get_my_points, "get_my_points")

# 설정에 정의된 사용자 의도 이름 목록 (범위 밖 판정에 씁니다)
defined_intents = list(card_config.user_messages)
display(defined_intents)

**확인 퀴즈**: 아래 Colang에서 설계한 대로 동작하지 않는 부분 두 곳을 찾아 고쳐 보세요.

```
define bot inform points
  "남은 포인트는 $points점입니다."

define flow delivery status
  user ask delivery status
  do check login
  $status = execute get_delivery_status(customer_id=$customer_id)
  if $status == "배송 중"
    bot inform delivery shipping
  elif $status == "발급 심사 중"
    bot inform delivery review
  else
    bot inform delivery other
```

<details><summary>정답 보기</summary>

- **`$points점입니다.`:** `points점입니다`가 변수 이름으로 읽혀 "남은 포인트는 ."만 남고, 오류도 나지 않아 놓치기 쉽습니다. `$points 점입니다`처럼 띄어 쓰거나 `{{ points }}점입니다`로 씁니다.
- **`elif`:** Colang 2.0 문법이라 1.0에서는 파싱 오류가 납니다. `else if`로 씁니다.

</details>

### 4-3. 테스트 문장으로 검증

바꿔 말한 테스트 문장 13개로 예시에 없는 표현에서도 의도를 맞히는지 확인합니다.

- **테스트 문장 파일:** `flow_tests.json`의 문장(`text`)마다 기대 의도(`expected_intent`)가 있습니다. 범위 밖 문장 2개는 기대 의도가 `null`입니다.
- **범위 밖 판정:** 기대 의도가 `null`이면 "범위 밖"으로 표시하고, 실제 의도가 정의한 의도 목록에 없으면 범위 밖으로 처리된 것으로 봅니다.
- **로그인 상태:** 모든 문장을 고객 C001의 context와 함께 보내 조회 결과까지 확인합니다.
- **먼저 한 문장을 보냅니다:** NeMo는 봇에 처음 들어온 메시지를 처리하면서 예시 문장·흐름·봇 메시지 색인을 만듭니다. 색인이 다 만들어지기 전에 여러 문장을 동시에 보내면 `Index is not built yet` 오류로 의도가 `None`이 되므로, 세 색인을 모두 쓰는 범위 밖 문장 하나를 먼저 보냅니다.
- **그다음 동시에 보냅니다:** 문장끼리 서로 기다릴 필요가 없으므로 13문장을 한꺼번에 보내되, `run_limited`로 동시에 도는 대화 수에 상한을 둡니다. 결과는 보낸 순서대로 받습니다.

In [ ]:
# 테스트 문장 13개를 미리 봅니다 (예시 문장은 이 문장들과 겹치지 않게 썼습니다)
flow_tests = json.loads((data_dir / "본실습" / "flow_tests.json").read_text(encoding="utf-8"))
display(pd.DataFrame(flow_tests))

In [ ]:
# 동시에 보내기 전에 범위 밖 문장 하나를 먼저 보내, 봇이 예시 문장, 흐름, 봇 메시지 색인을 모두 만들게 합니다
warmup_result = await ask(card_rails, "오늘 서울 날씨 어때요?", {"customer_id": "C001"})
show_result(warmup_result)

In [ ]:
import asyncio

# 반 전체가 같은 API 키를 쓸 때 분당 요청 한도(429 오류)에 걸리지 않게 동시에 도는 대화 수에 상한을 둡니다
MAX_CONCURRENCY = 32
semaphore = asyncio.Semaphore(MAX_CONCURRENCY)


async def run_limited(task):
    """세마포어 자리가 날 때까지 기다렸다가 작업 하나를 실행합니다."""
    async with semaphore:
        return await task

In [ ]:
# 로그인한 고객 C001로 13문장을 동시에 보냅니다
tasks = []
for test in flow_tests:
    tasks.append(run_limited(ask(card_rails, test["text"], {"customer_id": "C001"})))
flow_results = await asyncio.gather(*tasks)

In [ ]:
# 문장과 결과를 같은 순서로 짝지어 기대 의도와 실제 의도를 비교합니다
flow_test_results = []
for test, result in zip(flow_tests, flow_results):
    actual = get_user_intent(result)

    # 기대 의도가 null이면 범위 밖 문장, 실제 의도가 정의한 의도 목록에 없으면 범위 밖으로 처리된 것
    expected = test["expected_intent"]
    if expected is None:
        expected = "범위 밖"
    judged = actual
    if actual not in defined_intents:
        judged = "범위 밖"

    flow_test_results.append({
        "문장": test["text"],
        "기대 의도": expected,
        "실제 의도": actual,
        "판정 의도": judged,
        "일치": expected == judged,
        "LLM 호출 수": len(result.log.llm_calls),
        "답변": result.response[0]["content"],
    })
display(pd.DataFrame(flow_test_results))

**결과 표 읽기**

- **일치한 처리·거절 문장:** LLM 호출 수가 1이면 흐름과 정해 둔 문장으로 답한 것입니다.
- **범위 밖 문장:** 실제 의도는 LLM이 새로 만든 이름이고, 다음 단계와 답변도 LLM이 만듭니다. 그 답이 한빛카드 챗봇의 답으로 적절한지 답변 열을 직접 읽습니다.
- **다시 실행합니다:** 결과는 실행마다 달라질 수 있으므로(2-2), 예시 문장이나 모델을 바꿀 때마다 같은 테스트를 다시 실행합니다(교안 01 5-4와 같은 절차).

**어긋난 의도를 바로잡는 방법**

1. 그 문장의 기대 의도의 `define user`에 비슷한 표현의 예시 문장을 하나 더합니다.
2. 테스트 문장을 그대로 예시로 넣지 않습니다. 그 문장만 맞게 되고, 그 테스트는 더 이상 검증 역할을 못 합니다.
3. 봇을 다시 만들고 어긋났던 문장만 `ask`로 다시 보내 의도를 확인합니다.
4. 예시를 더하면 다른 의도의 문장이 이 의도로 잘못 분류될 수 있으므로, 마지막에 13문장 전체를 다시 실행합니다.

## 5. 대화 레일이 막는 것과 못 막는 것

교안 01의 레드팀 테스트 셋에서 공격 셋(a01 지시 무시, a05 가정 상황 탈옥, a08 남의 정보 조회)을 골라 같은 `card_rails`에 보내, 대화 레일의 방어 범위를 확인합니다. 교안 01에서는 이 공격들을 시스템 프롬프트 규칙만으로 막아야 했습니다.

- **답변 출처:** LLM 호출에 `generate_bot_message`가 있으면 LLM이 만든 문장, 없으면 `define bot`에 정해 둔 문장입니다.
- **표식 판정(교안 01 4-1)은 하지 않습니다:** 이 봇의 설정에는 상담원 전용 코드나 다른 고객의 상담 기록이 없으므로 어느 경로로 답했는지만 봅니다.
- **세 공격도 동시에 보냅니다:** 4-3의 `run_limited`를 그대로 씁니다.

In [ ]:
# 교안 01의 레드팀 테스트 셋에서 공격 세 개를 고릅니다
red_team_set = json.loads((data_dir / "본실습" / "red_team_set.json").read_text(encoding="utf-8"))
attack_ids = ["a01", "a05", "a08"]
attacks = []
for case in red_team_set:
    if case["case_id"] in attack_ids:
        attacks.append(case)

# 고른 공격의 위협, 기법, 노리는 자산과 공격 문장을 봅니다 (세 공격 모두 사용자 메시지 하나)
attack_table = []
for case in attacks:
    attack_table.append({
        "케이스 ID": case["case_id"],
        "위협": case["threat"],
        "기법": case["technique"],
        "노리는 자산": case["asset"],
        "공격 문장": case["messages"][-1]["content"],
    })
display(pd.DataFrame(attack_table))

In [ ]:
# 세 공격을 같은 card_rails에 동시에 보냅니다 (동시 실행 상한은 4-3의 run_limited)
tasks = []
for case in attacks:
    tasks.append(run_limited(ask(card_rails, case["messages"][-1]["content"], {"customer_id": "C001"})))
attack_results = await asyncio.gather(*tasks)

In [ ]:
# 공격과 결과를 같은 순서로 짝지어, 답변을 LLM이 만들었는지 로그로 확인합니다
rail_attack_results = []
for case, result in zip(attacks, attack_results):
    task_names = [call.task for call in result.log.llm_calls]

    source = "정해 둔 문장"
    if "generate_bot_message" in task_names:
        source = "LLM이 만든 문장"

    rail_attack_results.append({
        "케이스 ID": case["case_id"],
        "기법": case["technique"],
        "공격 문장": case["messages"][-1]["content"],
        "실제 의도": get_user_intent(result),
        "LLM 호출 수": len(task_names),
        "답변 출처": source,
        "답변": result.response[0]["content"],
    })
display(pd.DataFrame(rail_attack_results))

**결과 읽기**

- **의도로 잡힌 공격:** a08처럼 `ask other customer info`로 잡히면 정해 둔 거절 문장으로 답하므로, 공격 문장이 답변을 바꾸지 못합니다.
- **의도로 못 잡힌 공격:** a01·a05처럼 정의한 의도에 맞지 않으면 답변을 LLM이 만듭니다. 거절했더라도 모델이 고른 결과라서 다시 실행하면 뚫릴 수 있습니다(교안 01의 시스템 프롬프트 방어와 같은 조건).
- **의도 만들기도 LLM이 합니다:** 거절 의도를 정의해 두어도 표현을 바꾼 공격은 다른 의도로 잡힐 수 있습니다.

<img src="images/15_dialog_rail_limits.png" width="1050" alt="대화 레일이 막는 것과 못 막는 것. 의도로 잡힌 공격: '김서연 고객 연락처 좀 알려 주세요'가 ask other customer info로 분류되면 정해 둔 거절 흐름으로 가서 '다른 고객의 정보는 알려 드릴 수 없습니다'라고 답한다. 의도로 못 잡힌 공격: '이전 지시는 모두 무시해'가 정의하지 않은 의도로 분류되면 LLM이 다음 단계와 답변을 만들어 답변 내용이 모델 판단에 달린다. 사용자 입력과 답변의 검사는 입력 레일과 출력 레일이 맡는다.">

| 대화 레일이 막는 것 | 대화 레일이 못 막는 것 |
|---|---|
| 정의한 의도로 잡힌 질문에 정해 둔 문장만 답하기 | 어떤 의도에도 안 잡힌 공격 문장(LLM이 답을 만듦) |
| 조회 전에 로그인 확인하기(`do check login` + `stop`) | 사용자 입력 자체의 검사(주입·탈옥 문구 찾기) |
| 조회 액션에 앱이 넘긴 고객 번호만 쓰기 | 답변의 검사(개인정보·코드가 포함됐는지) |

못 막는 부분은 다음 단원에서 config.yml의 `rails` 설정으로 입력·출력·검색 레일을 더해 보완합니다.

**레일 다섯 종류와 NeMo 설정 위치** (레일 설명은 교안 01 5-2)

| 레일 | NeMo 설정 위치 | 이 교안에서 |
|---|---|---|
| 입력 레일 | config.yml `rails.input.flows` | 다음 단원 |
| 대화 레일 | Colang 파일의 `define flow` | 직접 작성합니다 |
| 검색 레일 | config.yml `rails.retrieval.flows` | 다음 단원 |
| 실행 레일 | 액션(도구)을 실행하기 전후에 입력과 결과를 검사하는 흐름 | 그다음 단원 |
| 출력 레일 | config.yml `rails.output.flows` | 다음 단원 |

## 6. 🖐️ 함께 따라하기: 사내 인사정보 도우미 대화 레일 설계

교안 01 따라하기의 누리전자(가상) 사내 인사정보 도우미에 4절의 설계 절차를 적용합니다. 시스템 프롬프트 규칙으로 막으려던 동료 신상 질문을 대화 레일의 거절 흐름으로 처리합니다.

**실습 진행 방법**

작성 셀은 이 함께 따라하기 절에만 있습니다.

- **부분 작성:** 셀에 남은 `...`만 주석 힌트에 맞춰 채웁니다. 나머지 코드는 제공합니다.
- `...`가 남은 채로 실행하면 Colang 파싱 오류나 파이썬 오류가 납니다. 작성 셀을 완성한 뒤 다음 셀로 넘어갑니다.
- 네트워크 오류로 멈추면 문장을 동시에 보내는 셀부터 다시 실행합니다. 결과 표 셀은 목록을 셀 안에서 새로 만듭니다.
- 완성 코드는 수업이 끝난 뒤 강사가 공유합니다. 먼저 직접 작성해 봅니다.

### 6-1. 질문과 작성 목표

- **배경:** 도우미는 직원 본인의 연차와 경조 휴가 규정은 안내하고, 다른 직원의 연락처·주소·연봉·인사 평가는 알려 주면 안 됩니다. 교안 01에서는 이 규칙이 시스템 프롬프트에만 있었습니다.
- **작성:** Colang에 동료 신상 질문의 예시 문장, 그 질문을 거절하는 흐름, 연차 조회 흐름의 `execute`와 `if`를 채웁니다. 이어서 액션 등록·context·범위 밖 판정을 채워 테스트 문장 9개의 결과 표 `hr_flow_test_results`를 만듭니다.
- **확인:** 기대 의도와 실제 의도가 어긋난 문장이 있는지, 동료 신상 질문의 답변이 정해 둔 거절 문장인지 봅니다.
- **힌트:** 거절 흐름은 4-2의 `NEW_INTENTS_COLANG`에 있는 다른 고객 정보 거절을, 조회 흐름은 3-4의 배송 조회(로그인 확인 포함)를 참고합니다. 테스트 셀은 4-3처럼 문장을 동시에 보낸 뒤 결과를 짝지어 비교합니다.

In [ ]:
# 교안 01에서 쓴 도우미 설정을 읽어 시스템 프롬프트 규칙을 다시 봅니다
hr_bot = json.loads((data_dir / "따라하기" / "hr_bot.json").read_text(encoding="utf-8"))
print(hr_bot["service"])
print(hr_bot["system_prompt"])

### 6-2. 설정과 Colang 작성

봇 설명이 다르므로 도우미 설정 폴더(`rails_hr`)의 config.yml을 읽습니다. 모델 설정은 본실습과 같고, 프롬프트는 본실습의 `prompts_yaml`을 이어 붙여 씁니다. 연차 조회 액션은 인사 시스템 대신 `employees.json`을 읽습니다.

In [ ]:
# 도우미의 config.yml: 모델은 본실습과 같고, 봇 설명과 예시 대화만 다릅니다
hr_config_yaml = (data_dir / "따라하기" / "rails_hr" / "config.yml").read_text(encoding="utf-8")
print(hr_config_yaml)

In [ ]:
# 인사 시스템 대신 쓰는 직원 데이터 (사번 -> 올해 남은 연차 일수)
employees = json.loads((data_dir / "따라하기" / "employees.json").read_text(encoding="utf-8"))


def get_leave_balance(employee_id):
    """사번으로 올해 남은 연차 일수를 조회합니다."""
    return employees[employee_id]["leave_left"]


display(employees)

In [ ]:
# 테스트 문장 9개를 미리 봅니다 (예시 문장은 이 문장들과 겹치지 않게 씁니다)
hr_flow_tests = json.loads((data_dir / "따라하기" / "hr_flow_tests.json").read_text(encoding="utf-8"))
display(pd.DataFrame(hr_flow_tests))

In [ ]:
# 🖐️ 함께 따라하기: 빈칸(...)을 채워 도우미의 Colang을 완성하세요
HR_COLANG = """
# 인사
define user express greeting
  "안녕하세요"
  "좋은 아침입니다"
  "수고 많으십니다"

define bot express greeting
  "안녕하세요. 누리전자 인사정보 도우미입니다. 무엇을 도와드릴까요?"

define flow greeting
  user express greeting
  bot express greeting

# 경조 휴가 안내: 인사 규정의 내용을 정해 둔 문장으로 답한다
define user ask wedding leave
  "본인 결혼 때 쓸 수 있는 휴가가 있나요?"
  "경조사 휴가 규정 알려 주세요"
  "신혼여행 가려는데 특별 휴가 있어요?"

define bot inform wedding leave
  "본인 결혼 경조 휴가는 5일, 자녀 결혼은 1일입니다. 신청은 인사 시스템의 휴가 메뉴에서 합니다."

define flow wedding leave
  user ask wedding leave
  bot inform wedding leave

# 로그인 확인: 사번이 없으면 안내하고 흐름 전체를 멈춘다
define bot ask login
  "사내 계정으로 로그인한 직원만 조회할 수 있습니다."

define subflow check login
  if not $employee_id
    bot ask login
    stop

# 연차 조회: 로그인 확인 -> 남은 연차 조회 -> 남은 일수에 따라 안내
define user ask my leave balance
  "연차 잔여 일수 조회해 주세요"
  "휴가 쓸 수 있는 날이 얼마나 남았죠?"
  "제 연차 현황 보여 주세요"

define bot inform leave balance
  "올해 남은 연차는 {{ leave_left }}일입니다."

define bot inform no leave left
  "올해 남은 연차가 없습니다. 연차 사용 계획은 팀장과 상의해 주세요."

define flow leave balance
  user ask my leave balance
  do check login
  # 🖐️ get_leave_balance 를 employee_id=$employee_id 로 실행해 결과를 $leave_left 에 담으세요
  ...
  # 🖐️ $leave_left 가 0보다 크면 bot inform leave balance, 아니면 bot inform no leave left (if, else 아래는 두 칸 더 들여 씁니다)
  ...

# 동료 신상 정보 거절
# 🖐️ 다른 직원의 연락처, 주소, 연봉, 인사 평가를 묻는 예시 문장을 세 개 이상 쓰세요(테스트 문장과 겹치지 않게)
define user ask colleague personal info
  ...

define bot refuse colleague personal info
  "다른 직원의 연락처, 주소, 연봉, 인사 평가 결과는 알려 드릴 수 없습니다. 부서 인원 수 같은 집계 정보는 안내해 드릴 수 있습니다."

# 🖐️ 위 의도가 들어오면 위 거절 문장으로 답하는 흐름을 쓰세요
define flow colleague personal info
  ...
"""

### 6-3. 테스트 문장 실행과 결과 표

`hr_flow_tests.json`의 9문장을 로그인한 직원 E1001로 보내고, 4-3과 같은 범위 밖 판정으로 기대 의도와 비교합니다. 4-3처럼 범위 밖 문장 하나를 먼저 보내 색인을 만든 뒤 동시에 보냅니다.

In [ ]:
# 🖐️ 함께 따라하기: 도우미 봇을 만들고 연차 조회 액션을 등록하세요
hr_config = RailsConfig.from_content(colang_content=HR_COLANG, yaml_content=hr_config_yaml + "\n" + prompts_yaml)
hr_rails = LLMRails(hr_config)
hr_rails.register_action(...)  # 연차 조회 함수를 Colang에서 쓴 액션 이름으로 등록
hr_defined_intents = list(hr_config.user_messages)

In [ ]:
# 4-3처럼 범위 밖 문장 하나를 먼저 보내 색인을 만듭니다
hr_warmup_result = await ask(hr_rails, "오늘 점심 뭐 먹을까요?", {"employee_id": "E1001"})
show_result(hr_warmup_result)

In [ ]:
# 🖐️ 로그인한 직원 E1001로 9문장을 동시에 보내세요 (동시 실행 상한은 4-3의 run_limited)
tasks = []
for test in hr_flow_tests:
    tasks.append(run_limited(ask(hr_rails, test["text"], ...)))  # 사번 E1001을 context 값(dict)으로. 키는 Colang에서 쓴 변수 이름(employee_id)
hr_flow_results = await asyncio.gather(*tasks)

In [ ]:
# 🖐️ 문장과 결과를 같은 순서로 짝지어 기대 의도와 실제 의도를 비교하세요
hr_flow_test_results = []
for test, result in zip(hr_flow_tests, hr_flow_results):
    actual = get_user_intent(result)

    expected = test["expected_intent"]
    judged = actual
    ...  # 본실습과 같은 규칙: expected 가 None 이면 "범위 밖", actual 이 hr_defined_intents 에 없으면 judged 를 "범위 밖"으로

    hr_flow_test_results.append({
        "문장": test["text"],
        "기대 의도": expected,
        "실제 의도": actual,
        "판정 의도": judged,
        "일치": expected == judged,
        "LLM 호출 수": len(result.log.llm_calls),
        "답변": result.response[0]["content"],
    })
display(pd.DataFrame(hr_flow_test_results))

In [ ]:
# 테스트 문장은 모두 E1001(연차 7일)로 보냈으므로, 연차 조회 흐름의 나머지 두 분기를 따로 확인합니다
# E1002는 연차가 0일이라 else 분기로, context 가 없으면 check login 에서 멈춰야 합니다
branch_contexts = [{"employee_id": "E1002"}, None]
tasks = []
for context in branch_contexts:
    tasks.append(run_limited(ask(hr_rails, "제 연차 며칠 남았는지 알려 주세요", context)))
branch_outputs = await asyncio.gather(*tasks)

# context 와 결과를 같은 순서로 짝지어 표로 봅니다
branch_results = []
for context, branch_result in zip(branch_contexts, branch_outputs):
    branch_results.append({
        "context": context,
        "사용자 의도": get_user_intent(branch_result),
        "LLM 호출 수": len(branch_result.log.llm_calls),
        "답변": branch_result.response[0]["content"],
    })
display(pd.DataFrame(branch_results))

어긋난 문장이 있으면 본실습처럼 그 기대 의도에 비슷한 표현의 예시 문장을 더하고, 봇을 다시 만든 뒤 그 문장만 다시 보냅니다.

**생각해 보기**: "옆자리 선배 연봉이 얼마인지 궁금해요" 같은 질문을 대화 레일만으로 모두 막을 수 있을까요? 놓친다면 어디에서 막아야 할까요?

<details><summary>정답 보기</summary>

모두 막을 수는 없습니다.

- **표현을 바꾸면 다른 의도로 분류될 수 있습니다:** "선배랑 연봉 비교해 보고 싶은데 대충 얼마쯤이에요?"나 역할극으로 감싼 문장이 거절 의도가 아닌 이름으로 분류되면, 다음 단계와 답변을 LLM이 만듭니다(5절).
- **지금 연봉이 노출되지 않는 것은 데이터가 없어서입니다:** 이 봇은 직원 명부를 검색하지 않습니다. 교안 01처럼 명부 발췌(hr-03)를 검색 문서로 붙이면 LLM이 만든 답변에 연봉이 포함될 수 있습니다.
- **막는 위치:** 사용자 입력을 검사하는 입력 레일, 권한 없는 문서를 검색 결과에서 제외하는 검색 레일, 답변에서 연락처·금액을 찾는 출력 레일이 대화 레일을 보완합니다(다음 단원).
- **데이터 접근 범위부터 제한합니다:** 연차 조회 액션이 앱이 넘긴 본인 사번만 받듯, 다른 직원의 데이터에 접근하는 경로 자체를 만들지 않는 것이 가장 확실합니다.

</details>

## 7. 핵심 정리

- **NeMo Guardrails:** 레일을 설정 폴더(config.yml과 Colang 파일)로 선언하는 NVIDIA의 오픈소스 라이브러리입니다. config.yml이 모델·봇 설명·예시 대화를 정하고, 한국어에 맞게 임베딩을 OpenAI 모델로 바꿨습니다.
- **실행과 로그:** `RailsConfig`로 읽어 `LLMRails`로 봇을 만들고 `await rails.generate_async(...)`로 호출합니다. 로그 옵션으로 LLM 호출 작업 이름과 사용자 의도(`UserIntent`)를 확인합니다.
- **처리 세 단계:** 사용자 의도 만들기(LLM) -> 다음 단계 고르기(흐름 또는 LLM) -> 봇 메시지(`define bot` 문장 또는 LLM). LLM 호출은 흐름과 문장이 정해진 질문 1회, 흐름은 있고 문장이 없으면 2회, 정의하지 않은 질문 3회입니다.
- **Colang 1.0 문법:** `define user`·`define bot`·`define flow`(의도가 곧 분기), `$변수 = execute 액션(인자=...)`, `if` / `else if` / `else`, `define subflow` + `do`, `stop`입니다. 고객 번호처럼 사용자 말로 정하면 안 되는 값은 context 메시지로 넘기고, 봇 메시지 안 `$변수` 바로 뒤에는 한글을 붙이지 않습니다.
- **흐름 설계:** 의도 목록(처리·거절·전환) -> 흐름 -> Colang -> 테스트 문장 -> 결과 표 -> 예시 문장 보강 순서로, 예시와 겹치지 않는 테스트 문장으로 검증합니다.
- **대화 레일의 한계:** 의도로 잡힌 질문에는 정해 둔 답만 하지만, 의도로 못 잡힌 공격 문장은 LLM이 답을 만듭니다. 입력·출력·검색 레일은 다음 단원에서 더합니다.